In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, ArrayType
from pyspark.sql.functions import col, from_json
from pyspark.ml.fpm import FPGrowth

# ---------------------------------------------------------
# 1. KHỞI TẠO SPARK SESSION
# ---------------------------------------------------------
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
# Nếu chạy qua terminal: spark-submit --packages org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0 2_kafka_streaming_fpgrowth.py

spark = SparkSession.builder \
    .appName("MarketBasket_Kafka_Streaming") \
    .master("local[*]") \
    .config("spark.driver.memory", "16g") \
    .config("spark.sql.shuffle.partitions", "32") \
    .getOrCreate()

spark.sparkContext.setLogLevel("WARN")

# ---------------------------------------------------------
# 2. LOAD DỮ LIỆU ĐÃ TIỀN XỬ LÝ & TRAIN MODEL (OFFLINE BATCH)
# ---------------------------------------------------------
PARQUET_PATH = "data/history_data.parquet"
print(f"Đang đọc dữ liệu lịch sử từ {PARQUET_PATH}...")

# Load file parquet đã qua tiền xử lý (Lúc này cột Product đã chuẩn format Array)
history_df = spark.read.parquet(PARQUET_PATH)

print("Đang huấn luyện mô hình FP-Growth (Có thể mất vài phút tùy lượng data)...")
# Cấu hình minSupport và minConfidence tùy thuộc vào độ lớn dữ liệu của bạn
fpGrowth = FPGrowth(itemsCol="Product", minSupport=0.05, minConfidence=0.2)
model = fpGrowth.fit(history_df)

print("Huấn luyện xong! Các luật kết hợp (Top 20):")
model.associationRules.show(20, truncate=False)


# ---------------------------------------------------------
# 3. KẾT NỐI KAFKA VÀ ĐỌC LUỒNG STREAMING
# ---------------------------------------------------------
print("Đang lắng nghe Kafka stream trên topic 'sales_stream'...")

# Schema hứng dữ liệu từ Kafka (Client/Web đẩy lên)
# Định dạng JSON kỳ vọng: {"Transaction_ID": "T999", "Product": ["Sữa", "Táo"]}
json_schema = StructType([
    StructField("Transaction_ID", StringType(), True),
    StructField("Product", ArrayType(StringType()), True)
])

kafka_df = spark.readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "localhost:9092") \
    .option("subscribe", "sales_stream") \
    .option("startingOffsets", "latest") \
    .load()

# Parse JSON
parsed_stream_df = kafka_df \
    .selectExpr("CAST(value AS STRING) as json_string") \
    .select(from_json(col("json_string"), json_schema).alias("data")) \
    .select("data.Transaction_ID", "data.Product")


# ---------------------------------------------------------
# 4. ÁP DỤNG LUẬT KẾT HỢP (REAL-TIME RECOMMENDATION)
# ---------------------------------------------------------
# Sinh ra cột "prediction" chứa danh sách gợi ý mua kèm
predictions_stream = model.transform(parsed_stream_df)


# ---------------------------------------------------------
# 5. XUẤT KẾT QUẢ
# ---------------------------------------------------------
# Xuất ra console để kiểm tra luồng chạy thực tế
query = predictions_stream \
    .select("Transaction_ID", "Product", "prediction") \
    .writeStream \
    .outputMode("append") \
    .format("console") \
    .option("truncate", "false") \
    .start()

query.awaitTermination()